# 02 · Benchmark — how well can the salary range be predicted, and from what?

**Questions this notebook answers:**

1. How far can we get beyond the metadata-only floor from the EDA (R² 0.48, log MAE 0.22)?
2. What does each **mask pattern** cost — description only, metadata only, title only — using
   one model trained with block dropout (ADR 0007)?
3. What does block dropout cost on **fully specified** inputs? That number decides whether
   ADR 0007 stands or is superseded.
4. How much of the accuracy comes from **recognising the employer**? (company-held-out folds, ADR 0006)
5. What do per-block attributions look like on real postings?

**Protocol.** Everything starts from the derived table (ADR 0011). Model selection uses the five
collapse-grouped CV folds on the training split; the time holdout (newest 15%) is scored once at
the end; company-held-out folds are reported separately. The feature transformer is fitted inside
every fold because the company encoder uses the target (ADR 0010).

**Models, in order:** median by category (baseline) → ridge on metadata blocks → ridge on all
blocks → LightGBM on all blocks → LightGBM with block dropout.

In [ ]:
import time
import warnings

import lightgbm as lgb
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score

from salary_scout.data import REPO_ROOT
from salary_scout.dataset import load_derived
from salary_scout.features import (
    BLOCK_NAMES,
    FeatureBlocks,
    check_salary_leakage,
    dropout_blocks,
    mask_blocks,
)

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

# Chart chrome, same as the EDA notebook: one hue for magnitude, fixed categorical order.
BLUE, ORANGE, AQUA, YELLOW, MAGENTA = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"
SERIES = [BLUE, ORANGE, AQUA, YELLOW, MAGENTA]
plt.rcParams.update({
    "figure.dpi": 110, "figure.figsize": (9, 4),
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": "#52514e",
    "xtick.color": "#898781", "ytick.color": "#898781",
    "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.titlelocation": "left", "axes.titleweight": "bold", "axes.titlesize": 11,
    "axes.prop_cycle": plt.cycler(color=SERIES),
})
usd_fmt = mtick.FuncFormatter(lambda v, _: f"${v/1000:.0f}k")
RESULTS_DIR = REPO_ROOT / "docs" / "results"
RESULTS_DIR.mkdir(exist_ok=True)
SEED = 0

## 1. Data and splits

In [ ]:
df = load_derived()
train = df[df["split"] == "train"].reset_index(drop=True)
test = df[df["split"] == "test"].reset_index(drop=True)
leak = check_salary_leakage(df)
print(f"{len(df):,} rows; train {len(train):,}, test {len(test):,}; residual salary leakage {leak:.2%}")
print("publish window  train:", train["publish_ts"].min().date(), "→", train["publish_ts"].max().date(),
      "  test:", test["publish_ts"].min().date(), "→", test["publish_ts"].max().date())
pd.DataFrame({
    "rows": train.groupby("cv_fold").size(),
    "collapse groups": train.groupby("cv_fold")["collapse_key"].nunique(),
    "companies": train.groupby("cv_fold")["company_name"].nunique(),
    "rows (company folds)": train.groupby("company_fold").size(),
    "companies (company folds)": train.groupby("company_fold")["company_name"].nunique(),
}).rename_axis("fold")

## 2. Metrics and mask patterns

Both targets are predicted in log space (ADR 0003) and the range is rebuilt from them, so
`min ≤ max` always holds. Reported per model and pattern:

- `log_mae`, `r2` — on `log_mid`, comparable to the EDA floor
- `mae_usd`, `mape` — on the midpoint in dollars
- `overlap` — share of postings whose true range overlaps the predicted range
- `spread_mae` — on `log_spread`, the band width

A mask pattern is a set of blocks removed at prediction time with `mask_blocks`; the model
never sees the underlying columns and the block's presence indicator is zero.

In [ ]:
MASKS = {
    "full": [],
    "description_only": ["title", "role_meta", "location", "company"],
    "metadata_only": ["title", "description"],
    "title_only": ["description", "role_meta", "location", "company"],
}
METRICS = ["log_mae", "r2", "mae_usd", "mape", "overlap", "spread_mae"]


def reconstruct_range(log_mid, log_spread):
    mid = np.exp(log_mid)
    spread = np.exp(np.clip(log_spread, 0, None))
    lo = 2 * mid / (1 + spread)
    return lo, lo * spread, mid


def score(frame, log_mid_pred, log_spread_pred):
    lo, hi, mid = reconstruct_range(log_mid_pred, log_spread_pred)
    y = frame["log_mid"].to_numpy()
    return {
        "log_mae": np.mean(np.abs(log_mid_pred - y)),
        "r2": r2_score(y, log_mid_pred),
        "mae_usd": np.mean(np.abs(mid - frame["mid"].to_numpy())),
        "mape": np.mean(np.abs(mid - frame["mid"].to_numpy()) / frame["mid"].to_numpy()),
        "overlap": np.mean((lo <= frame["yearly_max_compensation"].to_numpy())
                           & (hi >= frame["yearly_min_compensation"].to_numpy())),
        "spread_mae": np.mean(np.abs(np.clip(log_spread_pred, 0, None) - frame["log_spread"].to_numpy())),
    }

## 3. Models

Every model predicts both targets. The tree models are fixed-size (no early stopping on the
evaluation fold) so nothing tunes itself on the data being scored.

- **median_by_category**: median `log_mid` of the training rows with the same
  `job_category` and `seniority_level`; global median when either is masked.
- **ridge_metadata**: ridge on the feature matrix with the two text blocks zeroed. Identical
  to fitting on a metadata-only transform, but reuses the group-cross-fitted company encoding.
- **ridge_all_blocks**: ridge on the full matrix (~464k sparse columns).
- **lgbm_all_blocks**: LightGBM, 800 trees, no dropout.
- **lgbm_block_dropout**: same LightGBM, trained on the training rows plus one copy of each
  row with random blocks masked (each block with probability 0.3, at least one block kept).
  Its feature transformer is fitted on that augmented frame so the company encoding stays
  cross-fitted for the copies too.

In [ ]:
DROPOUT_P = 0.3
LGBM_PARAMS = dict(n_estimators=800, learning_rate=0.05, num_leaves=63, colsample_bytree=0.3,
                   min_child_samples=20, subsample=0.8, subsample_freq=1, verbose=-1, random_state=SEED)


class CategoryMedian:
    keys = ["job_category", "seniority_level"]

    def fit(self, frame):
        self.global_mid_ = frame["log_mid"].median()
        self.global_spread_ = frame["log_spread"].median()
        self.table_ = frame.groupby(self.keys)["log_mid"].median().rename("m").reset_index()
        return self

    def predict(self, frame):
        merged = frame[self.keys].merge(self.table_, on=self.keys, how="left")
        mid = merged["m"].fillna(self.global_mid_).to_numpy()
        return mid, np.full(len(frame), self.global_spread_)


class PairModel:
    def __init__(self, factory):
        self.factory = factory

    def fit(self, X, log_mid, log_spread):
        self.mid_ = self.factory().fit(X, log_mid)
        self.spread_ = self.factory().fit(X, log_spread)
        return self

    def predict(self, X):
        return self.mid_.predict(X), self.spread_.predict(X)


def zero_blocks(X, fb, blocks):
    keep = np.ones(X.shape[1])
    for b in blocks:
        keep[fb.block_slices_[b]] = 0
    return (X @ sp.diags(keep)).tocsr()


def fit_models(fit_df, seed=SEED):
    timings = {}
    t = time.time()
    fb = FeatureBlocks(random_state=seed)
    X = fb.fit_transform(fit_df, fit_df["log_mid"])
    y_mid, y_spr = fit_df["log_mid"].to_numpy(), fit_df["log_spread"].to_numpy()
    timings["features"] = time.time() - t

    models = {}
    models["median_by_category"] = CategoryMedian().fit(fit_df)
    t = time.time()
    models["ridge_metadata"] = PairModel(lambda: Ridge(alpha=1.0)).fit(
        zero_blocks(X, fb, ["title", "description"]), y_mid, y_spr)
    models["ridge_all_blocks"] = PairModel(lambda: Ridge(alpha=1.0)).fit(X, y_mid, y_spr)
    timings["ridge"] = time.time() - t
    t = time.time()
    models["lgbm_all_blocks"] = PairModel(lambda: lgb.LGBMRegressor(**LGBM_PARAMS)).fit(X, y_mid, y_spr)
    timings["lgbm"] = time.time() - t

    t = time.time()
    aug = pd.concat([fit_df, dropout_blocks(fit_df, p=DROPOUT_P, rng=seed)], ignore_index=True)
    fb_do = FeatureBlocks(random_state=seed)
    X_aug = fb_do.fit_transform(aug, aug["log_mid"])
    models["lgbm_block_dropout"] = PairModel(lambda: lgb.LGBMRegressor(**LGBM_PARAMS)).fit(
        X_aug, aug["log_mid"].to_numpy(), aug["log_spread"].to_numpy())
    timings["lgbm_dropout"] = time.time() - t
    return {"fb": fb, "fb_do": fb_do, "models": models, "timings": timings}


def evaluate(fitted, eval_df, protocol, fold):
    rows = []
    for pattern, blocks in MASKS.items():
        ev = mask_blocks(eval_df, blocks)
        X, X_do = fitted["fb"].transform(ev), fitted["fb_do"].transform(ev)
        for name, m in fitted["models"].items():
            if name == "median_by_category":
                mid, spr = m.predict(ev)
            else:
                mid, spr = m.predict(X_do if name == "lgbm_block_dropout" else X)
            rows.append({"protocol": protocol, "fold": fold, "model": name, "pattern": pattern,
                         **score(eval_df, mid, spr)})
    return pd.DataFrame(rows)


MODEL_ORDER = ["median_by_category", "ridge_metadata", "ridge_all_blocks", "lgbm_all_blocks", "lgbm_block_dropout"]
PATTERN_ORDER = list(MASKS)


def summarise(res, metric, agg="mean"):
    t = (res.pivot_table(index="model", columns="pattern", values=metric, aggfunc=agg)
            .reindex(index=MODEL_ORDER, columns=PATTERN_ORDER))
    t.columns.name = f"{metric} ({agg})"
    return t

## 4. Grouped 5-fold cross-validation (training split)

In [ ]:
cv_results = []
for fold in range(5):
    t0 = time.time()
    fit_df = train[train["cv_fold"] != fold].reset_index(drop=True)
    val_df = train[train["cv_fold"] == fold].reset_index(drop=True)
    fitted = fit_models(fit_df)
    cv_results.append(evaluate(fitted, val_df, "grouped_cv", fold))
    tm = fitted["timings"]
    print(f"fold {fold}: fit {len(fit_df):,} / eval {len(val_df):,}  "
          f"features {tm['features']:.0f}s ridge {tm['ridge']:.0f}s lgbm {tm['lgbm']:.0f}s "
          f"lgbm+dropout {tm['lgbm_dropout']:.0f}s  total {time.time() - t0:.0f}s")
cv_results = pd.concat(cv_results, ignore_index=True)
cv_results.to_csv(RESULTS_DIR / "benchmark_grouped_cv.csv", index=False)

In [ ]:
print("log MAE on log_mid, mean over 5 grouped folds (EDA metadata floor: 0.22)")
display(summarise(cv_results, "log_mae").style.format("{:.3f}").background_gradient(cmap="Blues_r", axis=None))
print("fold-to-fold standard deviation of log MAE")
display(summarise(cv_results, "log_mae", "std").style.format("{:.3f}"))

In [ ]:
for metric, fmt in [("mape", "{:.1%}"), ("overlap", "{:.1%}"), ("r2", "{:.3f}"), ("spread_mae", "{:.3f}")]:
    print(metric, "— mean over folds")
    display(summarise(cv_results, metric).style.format(fmt))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.2))
tab = summarise(cv_results, "mape")
x = np.arange(len(MODEL_ORDER)); w = 0.2
for i, pattern in enumerate(PATTERN_ORDER):
    ax.bar(x + (i - 1.5) * w, tab[pattern].to_numpy(), width=w - 0.02, color=SERIES[i], label=pattern.replace("_", " "))
ax.set_xticks(x); ax.set_xticklabels([m.replace("_", "\n") for m in MODEL_ORDER])
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_ylabel("MAPE of midpoint (lower is better)")
ax.set_title("Grouped CV: typical error by model and mask pattern")
ax.set_ylim(0, tab.to_numpy().max() * 1.22)
ax.legend(frameon=False, ncol=4, loc="upper right")
ax.set_axisbelow(True); ax.grid(axis="x", visible=False)
plt.show()

## 5. What block dropout costs on full inputs

The single number ADR 0007 hinges on: LightGBM with dropout minus LightGBM without, on the
`full` pattern, per fold. A positive `log_mae` delta means dropout hurt. The same delta on the
masked patterns shows what dropout buys.

In [ ]:
pair = cv_results[cv_results["model"].isin(["lgbm_all_blocks", "lgbm_block_dropout"])]
wide = pair.pivot_table(index=["fold", "pattern"], columns="model", values=["log_mae", "mape", "overlap"])
delta = pd.DataFrame({
    "Δ log_mae (dropout − plain)": wide[("log_mae", "lgbm_block_dropout")] - wide[("log_mae", "lgbm_all_blocks")],
    "Δ mape": wide[("mape", "lgbm_block_dropout")] - wide[("mape", "lgbm_all_blocks")],
    "Δ overlap": wide[("overlap", "lgbm_block_dropout")] - wide[("overlap", "lgbm_all_blocks")],
})
dropout_cost = delta.groupby("pattern").agg(["mean", "std"]).reindex(PATTERN_ORDER)
display(dropout_cost.style.format("{:+.4f}"))
full_cost = delta.xs("full", level="pattern")["Δ log_mae (dropout − plain)"]
print(f"dropout cost on full inputs: Δ log MAE = {full_cost.mean():+.4f} ± {full_cost.std():.4f} "
      f"({full_cost.mean() / wide.xs('full', level='pattern')[('log_mae', 'lgbm_all_blocks')].mean():+.1%} relative)")

## 6. Time holdout (newest 15%)

Scored once, with every model refitted on the whole training split.

In [ ]:
t0 = time.time()
final = fit_models(train)
holdout = evaluate(final, test, "time_holdout", -1)
holdout.to_csv(RESULTS_DIR / "benchmark_time_holdout.csv", index=False)
print(f"refit on {len(train):,} rows in {time.time() - t0:.0f}s")
for metric, fmt in [("log_mae", "{:.3f}"), ("mape", "{:.1%}"), ("mae_usd", "${:,.0f}"), ("overlap", "{:.1%}"), ("r2", "{:.3f}")]:
    print(metric)
    display(summarise(holdout, metric).style.format(fmt))

## 7. Company-held-out folds

Same protocol, but folds are grouped by `company_name` so every evaluated employer is unseen.
The gap to §4 is the value of recognising the employer (ADR 0010 makes that one encoding).

In [ ]:
co_results = []
for fold in range(5):
    t0 = time.time()
    fit_df = train[train["company_fold"] != fold].reset_index(drop=True)
    val_df = train[train["company_fold"] == fold].reset_index(drop=True)
    co_results.append(evaluate(fit_models(fit_df), val_df, "company_heldout", fold))
    print(f"company fold {fold}: {time.time() - t0:.0f}s")
co_results = pd.concat(co_results, ignore_index=True)
co_results.to_csv(RESULTS_DIR / "benchmark_company_heldout.csv", index=False)

print("log MAE, company-held-out (mean over folds)")
display(summarise(co_results, "log_mae").style.format("{:.3f}"))
gap = summarise(co_results, "log_mae") - summarise(cv_results, "log_mae")
gap.columns.name = "Δ log_mae (company-held-out − grouped CV)"
print("cost of an unseen employer, by model and pattern")
display(gap.style.format("{:+.3f}"))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.8))
a = summarise(cv_results, "mape")["full"]; b = summarise(co_results, "mape")["full"]
y = np.arange(len(MODEL_ORDER))
ax.barh(y + 0.18, a.to_numpy(), height=0.34, color=BLUE, label="grouped CV (employers seen)")
ax.barh(y - 0.18, b.to_numpy(), height=0.34, color=ORANGE, label="company-held-out (employers unseen)")
ax.set_yticks(y); ax.set_yticklabels([m.replace("_", " ") for m in MODEL_ORDER]); ax.invert_yaxis()
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0)); ax.set_xlabel("MAPE of midpoint, full inputs")
ax.set_title("Seen vs unseen employers"); ax.legend(frameon=False, loc="lower right"); ax.grid(axis="y", visible=False)
plt.show()

## 8. Per-block attributions

TreeSHAP contributions from LightGBM (`pred_contrib=True`) summed per block with
`FeatureBlocks.sum_by_block`. Contributions are in log units, shown here as the multiplicative
effect on the midpoint (`exp(v) − 1`). Four test postings, dropout model, full inputs.

In [ ]:
fb_do, model_do = final["fb_do"], final["models"]["lgbm_block_dropout"]
rng = np.random.default_rng(SEED)
idx = np.sort(rng.choice(len(test), size=4, replace=False))
sample = test.iloc[idx].reset_index(drop=True)
X_s = fb_do.transform(sample)
contrib = model_do.mid_.booster_.predict(X_s, pred_contrib=True)
contrib = contrib.toarray() if hasattr(contrib, "toarray") else np.asarray(contrib)
base = contrib[0, -1]
by_block = fb_do.sum_by_block(contrib[:, :-1])
pred_mid, pred_spr = model_do.predict(X_s)
lo, hi, mid = reconstruct_range(pred_mid, pred_spr)
print(f"model baseline (expected log_mid): {np.exp(base):,.0f} USD")
summary = pd.DataFrame({
    "title": sample["title_clean"].str.slice(0, 50),
    "company": sample["company_name"], "state": sample["primary_state"],
    "true range": [f"${a:,.0f} – ${b:,.0f}" for a, b in zip(sample["yearly_min_compensation"], sample["yearly_max_compensation"])],
    "predicted range": [f"${a:,.0f} – ${b:,.0f}" for a, b in zip(lo, hi)],
})
display(summary)
display((np.exp(by_block) - 1).rename(columns=lambda c: f"{c} effect").style.format("{:+.1%}"))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(13, 3.6), sharex=True)
effects = np.exp(by_block) - 1
for i, ax in enumerate(axes):
    vals = effects.iloc[i].reindex(BLOCK_NAMES)
    colors = [BLUE if v >= 0 else ORANGE for v in vals]
    ax.barh(BLOCK_NAMES, vals.to_numpy(), color=colors, height=0.6)
    ax.axvline(0, color="#898781", linewidth=1)
    ax.invert_yaxis(); ax.grid(axis="y", visible=False)
    ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0, decimals=0))
    short = summary.loc[i, "title"][:28] + ("…" if len(summary.loc[i, "title"]) > 28 else "")
    ax.set_title(f"{short}\npredicted \\${mid[i]/1000:.0f}k · true \\${sample.loc[i, 'mid']/1000:.0f}k", fontsize=9.5)
fig.suptitle("Effect of each block on the predicted midpoint (blue raises, orange lowers)", x=0.01, ha="left", fontweight="bold", fontsize=11)
plt.tight_layout(); plt.show()

## 9. Findings

**Full inputs.** LightGBM on all blocks reaches log MAE 0.147 on the grouped CV and 0.144 on
the time holdout (MAPE 15.0% and 13.7%, R² 0.74 and 0.76). The predicted range overlaps the
true range for 84% of CV postings and 93% of holdout postings. The EDA floor was log MAE 0.22
and R² 0.48. Ridge on the same 464k-column matrix reaches 0.174; the rest is the tree model
combining text with metadata.

**Block dropout is free on full inputs and essential on masked ones.** The difference on full
inputs is −0.0002 ± 0.0009 log MAE, indistinguishable from zero. On masked inputs dropout cuts
log MAE from 0.247 to 0.190 (description only), 0.291 to 0.185 (metadata only) and 0.380 to
0.224 (title only). Without dropout, masked inputs are worse than the category-median baseline
in two of three patterns, and the ridge on all blocks collapses to R² −0.49 on metadata only.
**ADR 0007 is confirmed**; one model with block dropout serves every pattern.

**What each block is worth on its own** (dropout model, CV): metadata only 0.185 log MAE, 19%
MAPE, better than the EDA metadata ridge because the location and company blocks are richer;
description only 0.190; title only 0.224, so the title alone is as informative as the six
metadata fields were in the EDA.

**Recognising the employer contributes almost nothing.** Company-held-out folds score 0.148 on
full inputs against 0.147 with employers seen, and every model and pattern moves by at most
0.002. The company target encoding adds little beyond headcount, sector, organisation type and
the text. Two caveats: most employers have only a few postings, so their encoding sits near the
prior anyway, and descriptions often name the employer, so "unseen" applies to the encoding,
not to the text.

**Range width.** LightGBM predicts `log_spread` with MAE 0.115 against 0.157 for the constant
median, so the band width is partly predictable, but it remains the weaker of the two targets.

**Stability and drift.** Fold-to-fold standard deviation of log MAE is at most 0.006 for the
dropout model on every pattern; the no-dropout model varies up to 0.04 on title only. The time
holdout scores slightly better than the CV, so there is no sign of drift over the snapshot.

**Attributions.** Per-block TreeSHAP sums read naturally on the sampled postings: the
description and title typically carry the largest effects, and role metadata pulls the
estimate down for junior analyst roles. The explanation view in the apps can use this directly.

**For the next steps.** The service model is `lgbm_block_dropout` with the default feature
widths. The browser model needs a smaller matrix; the timing test showed that shrinking the
text hashes to 2^16 and 2^14 columns changes log MAE by less than 0.002, so ADR 0008 is
feasible without a second architecture.